In [5]:
import pandas as pd
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_from_disk
import os

In [6]:
# Load your trained model
peft_model_id = "/home/jovyan/Works/MSc_Thesis/mh_conversation/finetuned_model/llama3.2_3b_mh_conversation_v1" 

print("Loading trained model for prediction generation...")

# Load PEFT config to get base model info
config = PeftConfig.from_pretrained(peft_model_id)
print(f"Base model: {config.base_model_name_or_path}")

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map={"": 1},  # Use GPU 1
    torch_dtype=torch.bfloat16
)

# Load tokenizer from the adapter (contains special tokens)
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Resize model to accommodate new tokens
model.resize_token_embeddings(len(tokenizer))

# Load the adapter weights
model = PeftModel.from_pretrained(model, peft_model_id)
model.to(torch.bfloat16)
model.eval()

print("✅ Model loaded successfully!")

Loading trained model for prediction generation...
Base model: meta-llama/Llama-3.2-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.64it/s]


Tokenizer vocab size: 128256
✅ Model loaded successfully!


In [8]:
def test_model(patient_id, query, max_length=2000):
    """Test model with mental health query"""
    # Format exactly like training data
    messages = [
        {
            "role": "system",
            "content": """You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. \nThe assistant gives helpful, comprehensive, and appropriate answers to the user's questions."""
        },
        {
            "role": "user",
            "content": f"Query: {query}"
        }
    ]
    
    # Apply chat template using the LOADED tokenizer
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            # temperature=0.1,              
            # top_p=0.95,                 
            # repetition_penalty=1.0,     
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response_tokens = outputs[0][input_length:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=False)
    
    # Clean up the eos for clear response
    response = response.replace(tokenizer.eos_token, "").strip()
    
    return response

print("✅ test_model function defined!")

✅ test_model function defined!


In [9]:
def parse_dataset_sample(sample):
    """Parse your dataset sample to extract query and ground truth response"""
    return {
        'query': sample['query'],
        'ground_truth': sample['response'],
        'instruction': sample['instruction']
    }

print("✅ Dataset parser function defined!")

✅ Dataset parser function defined!


In [12]:
# Load test dataset
print("📊 Loading mental health test dataset...")

# Load your test dataset
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
dataset = load_from_disk(f"{dataset_based_path}/mh-mental-chat-16k_v1")
test_dataset = dataset["test"]

# Take a subset for evaluation (adjust size as needed)
# test_subset = test_dataset.select(range(10))  # Adjust size as needed
test_subset = test_dataset

print(f"🧪 Generating predictions for {len(test_subset)} test samples...")

# Prepare data storage
evaluation_data = []

for i, sample in enumerate(test_subset):
    # Parse the sample using your dataset structure
    parsed = parse_dataset_sample(sample)
    query = parsed['query']
    ground_truth = parsed['ground_truth']
    
    # Generate prediction using your model
    prediction = test_model("UID_TEST", query)
    
    # Store all data
    evaluation_data.append({
        'sample_id': i,
        'query': query,
        'ground_truth': ground_truth,
        'prediction': prediction,
        'instruction': parsed['instruction']
    })
    
    if (i + 1) % 10 == 0:
        print(f"✅ Processed {i + 1}/{len(test_subset)} samples...")

print(f"🎯 Generated {len(evaluation_data)} predictions")

📊 Loading mental health test dataset...
🧪 Generating predictions for 1606 test samples...
✅ Processed 10/1606 samples...
✅ Processed 20/1606 samples...
✅ Processed 30/1606 samples...
✅ Processed 40/1606 samples...
✅ Processed 50/1606 samples...
✅ Processed 60/1606 samples...
✅ Processed 70/1606 samples...
✅ Processed 80/1606 samples...
✅ Processed 90/1606 samples...
✅ Processed 100/1606 samples...
✅ Processed 110/1606 samples...
✅ Processed 120/1606 samples...
✅ Processed 130/1606 samples...
✅ Processed 140/1606 samples...
✅ Processed 150/1606 samples...
✅ Processed 160/1606 samples...
✅ Processed 170/1606 samples...
✅ Processed 180/1606 samples...


KeyboardInterrupt: 

In [11]:
# Create DataFrame and save to CSV
df = pd.DataFrame(evaluation_data)

# Create output directory if it doesn't exist
output_dir = "/home/jovyan/Works/MSc_Thesis/mh_conversation/model_evaluation/results"
os.makedirs(output_dir, exist_ok=True)

# Save to CSV
csv_filename = f"{output_dir}/mh_predictions_and_ground_truth.csv"
df.to_csv(csv_filename, index=False, encoding='utf-8')

print(f"✅ Saved evaluation data to: {csv_filename}")
print(f"📊 Dataset shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")

# Display sample
print("\n🔍 Sample data:")
print(df.head(2))

✅ Saved evaluation data to: /home/jovyan/Works/MSc_Thesis/mh_conversation/model_evaluation/results/mh_predictions_and_ground_truth.csv
📊 Dataset shape: (10, 5)
📋 Columns: ['sample_id', 'query', 'ground_truth', 'prediction', 'instruction']

🔍 Sample data:
   sample_id                                              query  \
0          0  Throughout our counseling session, I would lik...   
1          1  (sighing deeply) I've been feeling overwhelmed...   

                                        ground_truth  \
0  Managing stress-related challenges can be a di...   
1  (leaning forward, offering a sympathetic smile...   

                                          prediction  \
0  Managing stress can be a challenging experienc...   
1  I can see how challenging it is for you to reg...   

                                         instruction  
0  You are a helpful mental health counselling as...  
1  You are a helpful mental health counselling as...  


In [ ]:
# Verify the saved file
print("🔍 Verifying saved CSV file...")

# Load and check the saved data
loaded_df = pd.read_csv(csv_filename)
print(f"✅ Successfully loaded CSV with shape: {loaded_df.shape}")

# Show statistics
print(f"\n📊 Data Statistics:")
print(f"   Queries: {loaded_df['query'].notna().sum()}")
print(f"   Ground truths: {loaded_df['ground_truth'].notna().sum()}")
print(f"   Predictions: {loaded_df['prediction'].notna().sum()}")

print(f"\n💾 CSV file saved at: {csv_filename}")
print("🎉 Prediction generation complete!")